# 📗 질문에 필요한 문장을 추출해 답변합니다

## 컨텍스트 압축은 무엇인가요?

<strong>컨텍스트(Context)는 LLM이 질문에 답할 때 읽는 근거 자료입니다. 컨텍스트 압축(Contextual Compression)은 질문에 필요한 근거를 남겨 전달할 문맥을 줄이는 방법입니다.</strong> 이 교안에서는 리랭킹으로 고른 문서에서 LLM으로 관련 구간을 추출합니다.

압축 LLM도 원문을 읽고 발췌문을 출력하므로 추가 토큰을 사용합니다. 특히 짧은 청크 몇 개라면 압축을 생략하는 편이 나을 수 있습니다. 이 실습은 압축이 항상 유리하다고 가정하지 않습니다.

<img src="images/03_evidence_preservation.png" width="1100" alt="책 소개에서 질문에 필요한 학습 범위와 부정 표현을 유지하는 발췌 예시">

그림은 <strong>가상 도서 소개에서 수동으로 만든 발췌 예시</strong>입니다. 그림의 실행 기록 추적 여부 질문에서는 그 방법을 다루지 않는다는 근거를 남겨야 합니다.

## 실습 흐름

| 경로 | 같은 리랭킹 결과를 받은 뒤의 처리 | 집계하는 LLM 호출 |
|---|---|---|
| A: 리랭킹만 | 원문 → 답변 | 원문으로 답변 1회 |
| B: 리랭킹＋추출 | 원문 → LLM 추출 → 답변 | 문서별 추출＋추출문으로 답변 1회 |

같은 질문과 리랭킹 결과로 두 답변을 만듭니다. 추출문에서도 질문에 필요한 조건과 예외가 남아 있는지 원문을 읽어 확인합니다. 콜백으로 원문 답변, 추출, 추출문 답변의 토큰 사용량을 확인합니다.

## 오늘의 목표

- [ ] `LLMChainExtractor`로 질문에 필요한 구간을 추출할 수 있습니다.
- [ ] 원문과 추출문을 같은 답변 체인에 전달하고 결과를 확인할 수 있습니다.
- [ ] 콜백으로 답변과 추출의 토큰 사용량을 확인할 수 있습니다.
- [ ] 검색·리랭킹·추출을 하나의 파이프라인으로 연결할 수 있습니다.

시연은 <strong>가상 도서 51권의 소개·학습 내용</strong>, 따라하기는 <strong>고용노동부 매뉴얼</strong>입니다. 두 사례 모두 압축 이득이 없을 수 있습니다. 길고 복잡한 문서에서의 효과를 이 결과만으로 일반화하지 않습니다.

기본 비교는 사례당 추출 최대 3회＋답변 최대 2회입니다. 시연과 따라하기를 모두 완료하면 LLM 최대 10회이며 임베딩 요청은 별도입니다. 마지막 파이프라인 실습을 실행하면 새 질문 임베딩 1회와 추출 최대 3회가 추가됩니다.


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하세요. 경로·JSON 함수·원문 ID는 지난 교안의 방식을 이어갑니다.

따라하기는 책 소개와 다른 자료인 실제 고용노동부 매뉴얼 44절을 사용합니다. 본문과 PDF 위치는 지난 자료 그대로입니다.

교안 01에서 받은 Qwen3-Reranker-0.6B 파일을 재사용합니다. 아래에서는 현재 커널의 모델 객체만 준비합니다. CPU·입력 상한 1024토큰을 사용합니다.

아래 준비 코드는 한 셀입니다. `# ====` 구분선으로 역할을 나눴습니다. 위에서부터 실행하면 유틸리티와 시연·따라하기용 검색기가 준비됩니다. 이 셀을 다시 실행하면 문서를 다시 임베딩합니다.


In [ ]:
# ====================================================================
# 1) 라이브러리 가져오기
# 문서·표·토큰화·검색에 사용할 라이브러리를 가져옵니다.
# ====================================================================
# 본문과 출처를 같은 Document에 보관합니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# ====================================================================
# 2) 경로와 JSON 입출력
# material_dir를 기준으로 데이터를 읽고 결과를 저장하는 함수를 준비합니다.
# ====================================================================
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )

# ====================================================================
# 3) 환경변수와 모델 설정
# .env의 API 키를 읽고 문서와 질문에 공통으로 사용할 모델을 설정합니다.
# ====================================================================
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 이 구간은 모델 설정만 준비합니다. GPT 요청은 뒤의 체인 invoke에서 발생합니다.
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 아래 적재·검색 구간에서 요청합니다.")

# ====================================================================
# 4) 원문을 Document로 변환하는 함수
# 본문과 원문 ID·출처·검색 조건을 함께 보관합니다.
# ====================================================================
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 유지하고 필터 필드만 추가합니다.
    documents = []
    for record in records:
        document = Document(
            id=record["doc_id"],
            page_content=record["text"],
            metadata={"source_id": record["doc_id"], "title": record["title"],
                      "url": record["url"], **record["metadata"]},
        )
        documents.append(document)
    return documents

# ====================================================================
# 5) 시연 데이터 읽기
# 가상 도서 51권의 소개·학습 내용을 읽고 Document 목록으로 바꿉니다.
# ====================================================================
# 데이터 구조는 앞 5행으로 확인하고, 검색에는 전체 문서를 사용합니다.
records = read_json("demo_docs.json")
print("전체 문서 수:", len(records))
display(pd.DataFrame(records).head())
documents = make_documents(records)

# ====================================================================
# 6) 결과 출력과 한국어 토큰화
# show_results는 원문을 print로 보여 주고, kiwi_tokenize는 BM25에 쓸 토큰을 만듭니다.
# ====================================================================
def show_results(documents):
    """표시 순서와 원문 ID·메타데이터·본문 전체를 보여 줍니다."""
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    print("결과 문서 수:", len(documents))
    for order, doc in enumerate(documents, start=1):
        print(f"[{order}] 원문 ID:", doc.metadata["source_id"])
        print("메타데이터:", doc.metadata)
        # 본문은 별도 줄에 출력해 원문의 줄바꿈을 그대로 읽습니다.
        print("본문:")
        print(doc.page_content)
        print()

# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF에서 온 반각 가운뎃점(･)은 분석기가 앞뒤를 다른 낱말로 끊으므로 가운뎃점(·)으로 바꿉니다.
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    words = []
    for token in kiwi.tokenize(text):
        if token.tag.startswith("N") or token.tag in {"SL", "SN"}:
            words.append(token.form.lower())
    return words

# ====================================================================
# 7) 시연용 하이브리드 검색기
# 문서를 임베딩해 Chroma에 적재하고 BM25·Dense 검색기를 RRF로 연결합니다.
# ====================================================================
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day48_lesson02_books", embedding_function=embedding_model)
vector_store.reset_collection()
# 같은 원문 ID를 저장소 ID와 검색 결과 메타데이터에 함께 유지합니다.
added_ids = vector_store.add_documents(documents, ids=[doc.metadata["source_id"] for doc in documents])
print("day48_lesson02_books 적재 수:", len(added_ids))

# 각 검색기가 최대 5개씩 찾으므로 합친 후보는 최대 10개입니다. 리랭킹으로 3개를 선택합니다.
bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=5)
dense = vector_store.as_retriever(search_kwargs={"k": 5})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")

# ====================================================================
# 8) 따라하기용 PDF 데이터와 검색기
# 매뉴얼 44절을 별도 저장소에 적재하고 practice_hybrid를 준비합니다.
# ====================================================================
# PDF 자료도 앞 5행만 살펴보고 전체 문서를 검색에 사용합니다.
practice_records = read_json("practice_docs.json")
print("따라하기 전체 문서 수:", len(practice_records))
display(pd.DataFrame(practice_records).head())
practice_documents = make_documents(practice_records)

# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
practice_store = Chroma(collection_name="day48_lesson02_books_hr", embedding_function=embedding_model)
practice_store.reset_collection()
# 같은 원문 ID를 저장소 ID와 검색 결과 메타데이터에 함께 유지합니다.
added_ids = practice_store.add_documents(practice_documents, ids=[doc.metadata["source_id"] for doc in practice_documents])
print("day48_lesson02_books_hr 적재 수:", len(added_ids))

# 원리 비교 동안 같은 검색 단위와 후보 설정을 유지합니다.
practice_bm25 = BM25Retriever.from_documents(practice_documents, preprocess_func=kiwi_tokenize, k=5)
practice_dense = practice_store.as_retriever(search_kwargs={"k": 5})
practice_hybrid = EnsembleRetriever(
    retrievers=[practice_bm25, practice_dense], weights=[0.5, 0.5], c=60, id_key="source_id",
)

# ====================================================================
# 9) 리랭킹 구성요소와 모델
# 교안 01에서 받은 모델 파일로 현재 커널의 Cross-Encoder를 준비합니다.
# ====================================================================
# 공식 리랭커 인터페이스를 사용합니다.
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
# 문서 선택 뒤 질문 관련 문장만 추출하는 공식 구성요소입니다.
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, DocumentCompressorPipeline

# 교안 01에서 받은 모델 파일을 재사용해 현재 커널의 모델 객체를 준비합니다.
# 실습은 CPU와 입력 상한 1024토큰을 사용합니다. 긴 입력은 잘릴 수 있습니다.
cross_encoder = HuggingFaceCrossEncoder(
    model_name="Qwen/Qwen3-Reranker-0.6B",
    model_kwargs={"device": "cpu", "max_length": 1024},
)

# ====================================================================
# 10) 압축 실습에 사용할 원문 선택
# 질문으로 후보를 한 번 검색하고 리랭킹한 문서를 reranked에 보관합니다.
# ====================================================================
# 두 경로에서 이 질문과 선택 문서를 그대로 재사용합니다.
question = "에이전트가 도구를 잘못 고르거나 호출에 실패할 때 실행 기록을 추적해 원인을 찾는 방법을 배울 책을 찾아 주세요."
candidates = hybrid.invoke(question)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)
reranked = list(reranker.compress_documents(candidates, question))

show_results(reranked)


## 1. 비교할 원문을 준비합니다

준비 셀에서 선택한 `reranked`를 두 답변 경로에 그대로 사용합니다. 원문에는 <strong>도구 선택·호출 실패</strong>와 <strong>실행 기록을 추적하는 방법</strong>이 함께 있는지 읽어보세요. 뒤에서 추출한 문장과 답변도 이 원문에 비추어 확인합니다.

<img src="images/lesson02_section01_shared_context.png" width="1100" alt="같은 질문과 선택 문서를 유지하고, A에는 원문을, B에는 추출문을 전달합니다.">

같은 질문과 선택 문서를 유지하고, A에는 원문을, B에는 추출문을 전달합니다.


### 🖐️ 함께 따라하기: 재택근무 제한 조건의 원문을 준비합니다

같은 PDF 질문으로 검색·리랭킹한 원문을 두 답변 경로에서 재사용합니다. 검색 코드를 작성하고 출력된 원문에서 재택근무 제한 조건과 일부 요일·시간만 활용하는 대안을 읽으세요.


#### 1) 비교할 원문 검색과 선택

`practice_candidates`와 `practice_reranked`를 만드는 호출을 채우세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.
# 시연과 다른 PDF 원문에서 같은 비교를 수행합니다.
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"
practice_candidates = ...
practice_reranked = ...


#### 2) 선택 원문 읽기

제공 셀로 원문을 먼저 읽고 답변의 확인 기준을 정하세요.


In [ ]:
# [제공 코드]

# 답변을 보기 전에 선택된 원문부터 확인합니다.
show_results(practice_reranked)


### ✅ 바로 확인 퀴즈

원문에 답이 있었는데 리랭킹 결과에 그 원문이 없다면 추출 실패인가요?

<details><summary>정답 보기</summary>

아닙니다. 추출 전에 검색·문서 선택에서 빠진 것입니다. 두 경로가 같은 누락을 가진 상태임을 기록하고 압축 효과와 구분합니다.

</details>


## 2. 압축하지 않은 기준 답변을 먼저 만듭니다

A 경로는 `reranked` 원문을 그대로 사용합니다. 뒤의 B 경로도 <strong>같은 `answer_chain`에 같은 질문</strong>을 넣고 문맥만 바꿉니다. 출처 ID·제목·URL을 포함하는 형식도 같습니다.

### UsageMetadataCallbackHandler로 토큰 사용량을 기록합니다

<strong>콜백(callback)은 실행 중 특정 시점에 자동으로 호출되는 기능입니다.</strong> `UsageMetadataCallbackHandler`는 LLM 호출이 끝날 때 응답에 포함된 토큰 사용량을 받아 <strong>모델명별로 누적하는 객체</strong>입니다. 글자 수를 토큰 수로 추정하는 도구가 아닙니다.

사용 순서는 <strong>객체 만들기 → 호출에 연결하기 → 사용량 읽기</strong>입니다.

| 코드 | 역할 |
|---|---|
| `usage_a = UsageMetadataCallbackHandler()` | A 경로의 사용량 기록을 새로 시작합니다. |
| `config={"callbacks": [usage_a]}` | 체인 실행에 연결하여 내부 LLM의 사용량을 받습니다. |
| `usage_a.usage_metadata` | 호출이 끝난 뒤 모델명별 사용량 딕셔너리를 읽습니다. |

`usage_metadata`의 각 모델 값에는 `input_tokens`(입력), `output_tokens`(출력), `total_tokens`(합계)가 들어 있습니다. 실제 값과 모델명은 아래 출력으로 확인합니다. 모델이 사용량을 반환하지 않으면 기록이 비어 있을 수 있으며, 이를 0토큰으로 해석하지 않습니다.

`answer_chain`의 마지막 `StrOutputParser`는 답변을 문자열로 바꿉니다. 콜백은 그 전에 LLM 응답에서 사용량을 받아 별도로 보관하므로 <strong>답변 문자열과 사용량을 각각 확인</strong>할 수 있습니다.

이번 실습에서는 콜백에 기록된 토큰 사용량을 비교합니다. 객체를 만들거나 기록을 읽는 동작은 API를 호출하지 않습니다.

[공식 사용 예시](https://docs.langchain.com/oss/python/langchain/models#token-usage)

<img src="images/lesson02_section02_answer_usage.png" width="1100" alt="원문으로 답변을 생성하고, 콜백에서 입력·출력·전체 토큰 사용량을 읽습니다.">

원문으로 답변을 생성하고, 콜백에서 입력·출력·전체 토큰 사용량을 읽습니다.


In [ ]:
# 응답이 보고한 실제 토큰 사용량을 기록합니다.
from langchain_core.callbacks import UsageMetadataCallbackHandler


In [ ]:
# 출처와 본문을 함께 전달해 답변을 실제 원문으로 되짚을 수 있게 합니다.
def format_context(documents):
    """실제로 검색한 원문만 ID·제목·출처와 함께 답변 문맥으로 만듭니다."""
    context_parts = []
    for doc in documents:
        context = (
            f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n"
            f"출처: {doc.metadata['url']}\n{doc.page_content}"
        )
        context_parts.append(context)
    return "\n\n".join(context_parts)


In [ ]:
# 두 경로 모두 같은 프롬프트로 답변하며 근거의 원문 ID를 인용합니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 원문 근거로만 답하세요. 주장마다 [원문 ID]를 인용하세요. "
               "근거에 답이 없으면 확인할 수 없다고 말하세요. 자료의 빈 부분을 추측하지 마세요."),
    ("human", "질문: {question}\n\n원문 근거:\n{context}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()


In [ ]:
question = "에이전트가 도구를 잘못 고르거나 호출에 실패할 때 실행 기록을 추적해 원인을 찾는 방법을 배울 책을 찾아 주세요."

# A 경로: 원문을 그대로 전달합니다. 이 셀을 다시 실행하면 답변 요청이 추가됩니다.
usage_a = UsageMetadataCallbackHandler()
baseline_answer = answer_chain.invoke(
    {"question": question, "context": format_context(reranked)},
    config={"callbacks": [usage_a]},
)

print(baseline_answer)
print("A 토큰 사용량:", usage_a.usage_metadata)


PDF의 근로 규정에서 <strong>사용자</strong>는 회사(사업주), <strong>근로자</strong>는 직원입니다. 일상어의 '사용자'와 뜻이 달라 답변이 주체를 바꿔 쓰기 쉬우므로 용어 설명을 넣은 `practice_answer_chain`을 준비합니다. PDF의 A·B 모두 이 체인을 쓰며 원문은 바꾸지 않습니다.


In [ ]:
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"

# 두 경로 모두 같은 프롬프트로 답변하며 근거의 원문 ID를 인용합니다.
practice_answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "이 근로 규정에서 사용자란 회사(사업주), 근로자란 직원입니다. 제공된 원문 근거로만 답하세요. 주장마다 [원문 ID]를 인용하세요. "
               "근거에 답이 없으면 확인할 수 없다고 말하세요. 자료의 빈 부분을 추측하지 마세요."),
    ("human", "질문: {question}\n\n원문 근거:\n{context}"),
])
practice_answer_chain = practice_answer_prompt | llm | StrOutputParser()


### 🖐️ 함께 따라하기: PDF 원문으로 기준 답변을 만듭니다

원문을 넣는 A 경로의 콜백 생성과 답변 호출을 작성합니다. 같은 질문·체인을 뒤의 B 경로에서도 사용합니다. 결과 출력은 제공 셀을 실행하세요.


#### 1) 원문으로 기준 답변 생성

`practice_usage_a`를 만들고 그 콜백을 연결한 호출로 `practice_baseline_answer`를 받으세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.

# 뒤의 추출 경로와 질문·체인을 같게 둡니다.
practice_usage_a = ...
practice_baseline_answer = ...


#### 2) 답변과 사용량 확인

제공 셀을 실행해 답변과 토큰 사용량을 확인하세요.


In [ ]:
# [제공 코드]

# A 경로의 답변과 실제 사용량을 확인합니다.
print(practice_baseline_answer)
print("PDF A 토큰 사용량:", practice_usage_a.usage_metadata)


### ✅ 바로 확인 퀴즈

A와 B에서 검색을 각각 다시 하면 비교가 어려워지는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

검색 결과가 달라지면 답변 차이가 문서 선택 때문인지 압축 때문인지 구분하기 어렵습니다. 같은 `reranked`를 재사용합니다.

</details>


## 3. 질문에 필요한 구간을 추출해 답변합니다

<strong>`LLMChainExtractor`는 LLM으로 문서를 읽고, 질문에 필요한 구간만 뽑아내는 도구입니다.</strong> 리랭킹이 사용할 문서를 고르면, 추출기는 그 문서에서 답변에 필요한 내용을 골라냅니다.

별도의 LLM 모델이 아니라, 전달한 `llm`을 사용하는 도구입니다. <strong>추출용 기본 프롬프트가 내장되어 있어 따로 작성하지 않아도 됩니다.</strong> 기본 프롬프트는 “질문에 필요한 부분을 원문 그대로 추출하고, 관련 내용이 없으면 `NO_OUTPUT`을 반환하라”고 요청합니다.

<strong>원문 문서 + 질문 → 관련 구간 추출 → 추출문으로 답변 생성</strong>

### 두 단계로 사용합니다

1. <strong>`LLMChainExtractor.from_llm(llm)`</strong>: 사용할 LLM을 지정해 추출기를 준비합니다. 이때는 LLM을 호출하지 않습니다.
2. <strong>`extractor.compress_documents(reranked, question, ...)`</strong>: 문서마다 LLM을 호출해 질문과 관련된 구간을 추출합니다. 결과는 본문이 추출문으로 바뀐 `Document` 목록이며, 원문 ID 등 출처 정보는 유지됩니다.

관련 내용이 없으면 해당 문서가 결과에서 빠질 수 있습니다. 추출문은 답변을 만들 때 사용할 근거이므로, <strong>필요한 조건이나 예외가 빠지지 않았는지 원문과 대조하세요.</strong>

### 토큰 사용량을 함께 확인합니다

`usage_b`는 <strong>추출과 답변에 사용한 토큰을 기록하는 콜백</strong>입니다. 추출할 때 `callbacks=[usage_b]`를 전달하고, 뒤의 답변 생성에도 같은 콜백을 연결합니다. 그러면 <strong>추출 후에는 추출 토큰</strong>, <strong>답변 후에는 추출+답변의 누적 토큰</strong>을 확인할 수 있습니다.

<img src="images/lesson02_section03_extract_answer.png" width="1100" alt="질문에 필요한 구간을 추출하고 답변에 사용합니다. 같은 usage_b에 추출과 답변의 토큰 사용량을 누적합니다.">

질문에 필요한 구간을 추출하고 답변에 사용합니다. 같은 usage_b에 추출과 답변의 토큰 사용량을 누적합니다.


In [ ]:
# 객체 생성만으로는 API를 호출하지 않습니다.
extractor = LLMChainExtractor.from_llm(llm)


In [ ]:
question = "에이전트가 도구를 잘못 고르거나 호출에 실패할 때 실행 기록을 추적해 원인을 찾는 방법을 배울 책을 찾아 주세요."

# B 경로의 첫 단계입니다. 원문 reranked는 덮어쓰지 않습니다.
usage_b = UsageMetadataCallbackHandler()
compressed = list(extractor.compress_documents(reranked, question, callbacks=[usage_b]))

show_results(compressed)
print("B 추출까지의 토큰:", usage_b.usage_metadata)


출력된 추출문을 앞의 원문과 비교하세요. 질문에 필요한 조건·예외·부정 표현이 남았는지 확인합니다. 원문과 표현이 같다는 것만으로 필요한 근거가 모두 남았다고 판단하지 않습니다.


In [ ]:
# A에서 사용한 체인과 질문을 그대로 쓰고 문맥만 추출문으로 바꿉니다.
if compressed:
    compressed_answer = answer_chain.invoke(
        {"question": question, "context": format_context(compressed)},
        config={"callbacks": [usage_b]},  # 추출에 쓴 콜백을 그대로 연결합니다.
    )
else:
    compressed_answer = "전달할 근거가 없어 답변을 생성하지 않았습니다."

print(compressed_answer)
print("B 추출+답변 토큰:", usage_b.usage_metadata)


### 🖐️ 함께 따라하기: PDF 추출과 답변을 기록합니다

B 경로는 추출과 답변을 나누어 실행합니다. 두 호출에 같은 `practice_usage_b`를 연결하세요. 각 작성 셀 다음의 출력 제공 셀로 중간 결과를 확인합니다. 근거가 비면 답변을 생략하는 분기는 제공되어 있습니다.


#### 1) PDF 근거 추출

`practice_usage_b`를 만들고 콜백을 전달한 추출 호출로 `practice_compressed`를 받으세요.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"

# 같은 콜백에 PDF 추출과 답변의 토큰을 누적합니다.
practice_usage_b = ...
practice_compressed = ...


#### 2) 추출문과 중간 사용량 확인

제공 셀로 추출문과 추출까지의 토큰을 확인하세요. 답변 생성 전에 실행합니다.


In [ ]:
# [제공 코드]

# 답변을 만들기 전에 추출 결과와 이 시점의 사용량을 확인합니다.
show_results(practice_compressed)
print("PDF B 추출까지의 토큰:", practice_usage_b.usage_metadata)


#### 3) 추출문으로 답변 생성

같은 `practice_usage_b` 콜백을 연결해 `practice_compressed_answer`를 받으세요. 근거가 비면 답변을 생략합니다.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.

# 추출한 근거가 있을 때만 같은 콜백으로 답변을 생성합니다.
if practice_compressed:
    practice_compressed_answer = ...
else:
    practice_compressed_answer = "전달할 근거가 없어 답변을 생성하지 않았습니다."


#### 4) 답변과 누적 사용량 확인

제공 셀로 답변과 추출·답변에 사용한 누적 토큰을 확인하세요.


In [ ]:
# [제공 코드]

# B의 같은 콜백에 추출과 답변 사용량이 누적됩니다.
print(practice_compressed_answer)
print("PDF B 추출+답변 토큰:", practice_usage_b.usage_metadata)


### ✅ 바로 확인 퀴즈

추출 결과가 비어 B의 답변을 생략했다면 B의 토큰 사용량도 0인가요?

<details><summary>정답 보기</summary>

아닙니다. 답변 호출은 없지만 문서별 추출 호출은 이미 수행했습니다. `usage_b`에 기록된 추출 사용량은 그대로 남깁니다. 필요한 근거를 잃었다면 사용량이 적어도 좋은 결과가 아닙니다.

</details>


## 4. 검색·리랭킹·추출을 연결합니다

`DocumentCompressorPipeline(transformers=[reranker, extractor])`은 먼저 문서를 고른 뒤 관련 구간을 추출합니다. `ContextualCompressionRetriever`에 연결하면 `invoke` 한 번이 <strong>검색부터 다시 시작</strong>합니다. 새 질문에는 별도 콜백을 연결해 이번 실행의 추출 토큰을 확인합니다.

추출기를 먼저 두면 더 많은 후보를 LLM이 처리하고 리랭커도 원문 대신 추출문을 읽습니다. 이 실습은 리랭킹으로 문서를 먼저 줄이고 필요한 문장을 추출합니다.

<img src="images/lesson02_section04_pipeline.png" width="1100" alt="새 질문을 검색한 뒤 리랭커로 문서를 고르고, 추출기로 관련 구간을 남깁니다. 파이프라인의 순서는 리랭킹 다음 추출입니다.">

새 질문을 검색한 뒤 리랭커로 문서를 고르고, 추출기로 관련 구간을 남깁니다. 파이프라인의 순서는 리랭킹 다음 추출입니다.


In [ ]:
# 객체 연결만 준비합니다. 이 셀은 모델을 호출하지 않습니다.
compressor = DocumentCompressorPipeline(transformers=[reranker, extractor])
compression_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid, base_compressor=compressor,
)


In [ ]:
# 새 질문의 검색·리랭킹·추출을 실행합니다. 질문 임베딩 1회와 추출 최대 3회가 발생합니다.
next_question = "복잡한 요청을 여러 단계로 계획한 뒤 차례로 실행하는 에이전트를 만드는 방법을 배울 책을 찾아 주세요."
next_usage = UsageMetadataCallbackHandler()
next_compressed = compression_retriever.invoke(
    next_question, config={"callbacks": [next_usage]},
)
show_results(next_compressed)
print("새 질문의 추출 토큰:", next_usage.usage_metadata)


### 🖐️ 함께 따라하기: PDF 검색기를 파이프라인에 연결합니다

객체를 연결하는 핵심 코드만 작성하세요. 결과는 제공된 출력 셀로 확인하며 추가 모델 호출은 하지 않습니다.


#### 1) PDF 검색기와 압축 파이프라인 연결

`practice_hybrid`와 `compressor`를 연결해 `practice_compression_retriever`를 만드세요. 추가 invoke는 하지 않습니다.


In [ ]:
# [작성] ... 부분에 핵심 코드를 작성하세요.
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"

# 객체 연결은 요청을 발생시키지 않습니다.
practice_compression_retriever = ...


#### 2) 연결 결과 확인

제공 셀은 객체의 종류만 출력합니다. 검색·모델 요청은 발생하지 않습니다.


In [ ]:
# [제공 코드]

# 객체 생성 결과를 확인하며 모델을 호출하지 않습니다.
print("연결된 검색기:", type(practice_compression_retriever).__name__)
print("PDF 검색과 리랭킹·추출 연결 완료")


### ✅ 바로 확인 퀴즈

공식 파이프라인으로 연결했으니 두 방법의 비교도 다시 검색해서 해야 하나요?

<details><summary>정답 보기</summary>

아닙니다. 새 질문을 처리할 때는 연결한 검색기를 쓰지만, 압축 유무의 효과를 비교할 때는 동일한 선택 문서를 재사용합니다.

</details>

## 핵심 정리

- 같은 질문·선택 문서·답변 체인을 사용하고 전달하는 문맥만 원문에서 추출문으로 바꿉니다.
- `LLMChainExtractor`의 출력에서 조건·예외·부정 표현이 남았는지 원문을 읽어 확인합니다.
- A 콜백에는 원문 답변, B 콜백에는 추출과 추출문 답변의 토큰이 기록됩니다.
- 같은 콜백은 계속 누적됩니다. B 답변만 다시 실행하면 이전 답변의 사용량도 남으므로 다시 확인할 때는 B 추출부터 실행합니다.
- `DocumentCompressorPipeline`에 리랭커와 추출기를 순서대로 넣고 검색기에 연결합니다.

LV2에서는 사내 규정 문서로 원문 확인, 문장 추출, 답변 생성과 파이프라인 연결을 연습합니다.

참고: [LangChain 컨텍스트 압축](https://www.langchain.com/blog/improving-document-retrieval-with-contextual-compression), [사용량 추적](https://docs.langchain.com/oss/python/langchain/models#token-usage), [LLMChainExtractor](https://reference.langchain.com/python/langchain-classic/retrievers/document_compressors/chain_extract/LLMChainExtractor).


## 핵심 코드 이어서 보기

맨 위 <strong>준비 셀</strong>을 먼저 실행하세요. 기존 따라하기용 <strong>고용노동부 PDF 데이터 하나</strong>와 같은 질문으로 전체 흐름을 실행합니다. 준비된 `practice_hybrid`, `cross_encoder`, `llm`, `show_results`를 사용하므로 앞의 따라하기 빈칸을 풀지 않아도 됩니다.

<strong>검색·리랭킹 → 원문 답변 → 관련 구간 추출 → 추출문 답변 → 파이프라인 연결</strong> 순서입니다. 단계별 토큰 사용량을 확인하며, 마지막에는 같은 PDF 검색기와 질문으로 파이프라인을 실행합니다. 이 셀을 실행하면 질문 임베딩 2회와 LLM 최대 8회가 추가로 요청됩니다.


In [ ]:
# 1) 컨텍스트 압축과 토큰 기록에 필요한 구성요소
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker, LLMChainExtractor, DocumentCompressorPipeline

# 2) 기존 PDF 검색기로 후보를 찾고 리랭킹
practice_question = "고객사와 정기 회의가 있는 날에도 재택근무를 할 수 있나요?"
practice_candidates = practice_hybrid.invoke(practice_question)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)
practice_reranked = list(reranker.compress_documents(practice_candidates, practice_question))
show_results(practice_reranked)

# 3) 원문을 답변 체인에 전달할 형식 준비
# 출처와 본문을 함께 전달해 답변을 실제 원문으로 되짚을 수 있게 합니다.
def format_context(documents):
    """실제로 검색한 원문만 ID·제목·출처와 함께 답변 문맥으로 만듭니다."""
    context_parts = []
    for doc in documents:
        context = (
            f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n"
            f"출처: {doc.metadata['url']}\n{doc.page_content}"
        )
        context_parts.append(context)
    return "\n\n".join(context_parts)

# 4) PDF 근로 규정의 용어를 반영한 답변 체인
# 두 경로 모두 같은 프롬프트로 답변하며 근거의 원문 ID를 인용합니다.
practice_answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "이 근로 규정에서 사용자란 회사(사업주), 근로자란 직원입니다. 제공된 원문 근거로만 답하세요. 주장마다 [원문 ID]를 인용하세요. "
               "근거에 답이 없으면 확인할 수 없다고 말하세요. 자료의 빈 부분을 추측하지 마세요."),
    ("human", "질문: {question}\n\n원문 근거:\n{context}"),
])
practice_answer_chain = practice_answer_prompt | llm | StrOutputParser()

# 5) 원문으로 답변 생성
# 뒤의 추출 경로와 질문·체인을 같게 둡니다.
practice_usage_a = UsageMetadataCallbackHandler()
practice_baseline_answer = practice_answer_chain.invoke(
    {"question": practice_question, "context": format_context(practice_reranked)},
    config={"callbacks": [practice_usage_a]},
)

# 원문 답변과 토큰 사용량 확인
# A 경로의 답변과 실제 사용량을 확인합니다.
print(practice_baseline_answer)
print("PDF A 토큰 사용량:", practice_usage_a.usage_metadata)

# 6) 기본 프롬프트를 사용하는 추출기 준비
# 객체 생성만으로는 API를 호출하지 않습니다.
extractor = LLMChainExtractor.from_llm(llm)

# 7) 같은 질문에 필요한 구간 추출
# 같은 콜백에 PDF 추출과 답변의 토큰을 누적합니다.
practice_usage_b = UsageMetadataCallbackHandler()
practice_compressed = list(extractor.compress_documents(
    practice_reranked, practice_question, callbacks=[practice_usage_b],
))

# 추출문과 추출 토큰 확인
# 답변을 만들기 전에 추출 결과와 이 시점의 사용량을 확인합니다.
show_results(practice_compressed)
print("PDF B 추출까지의 토큰:", practice_usage_b.usage_metadata)

# 8) 추출문으로 답변 생성
# 추출한 근거가 있을 때만 같은 콜백으로 답변을 생성합니다.
if practice_compressed:
    practice_compressed_answer = practice_answer_chain.invoke(
        {"question": practice_question, "context": format_context(practice_compressed)},
        config={"callbacks": [practice_usage_b]},
    )
else:
    practice_compressed_answer = "전달할 근거가 없어 답변을 생성하지 않았습니다."

# 추출과 답변의 누적 토큰 확인
# B의 같은 콜백에 추출과 답변 사용량이 누적됩니다.
print(practice_compressed_answer)
print("PDF B 추출+답변 토큰:", practice_usage_b.usage_metadata)

# 9) 같은 PDF 검색·리랭킹·추출을 하나의 파이프라인으로 연결
compressor = DocumentCompressorPipeline(transformers=[reranker, extractor])
practice_compression_retriever = ContextualCompressionRetriever(
    base_retriever=practice_hybrid, base_compressor=compressor,
)

# 같은 질문으로 연결된 흐름을 실행합니다. 앞의 사용량과 섞이지 않게 새 콜백을 씁니다.
pipeline_usage = UsageMetadataCallbackHandler()
pipeline_compressed = practice_compression_retriever.invoke(
    practice_question, config={"callbacks": [pipeline_usage]},
)
show_results(pipeline_compressed)
print("파이프라인 추출 토큰:", pipeline_usage.usage_metadata)
